# 08 — Async & Batch Operations

This notebook covers high-throughput patterns:
- Async client: `aretain()`, `arecall()`, `areflect()`
- Concurrent operations with `asyncio.gather`
- Batch retains for bulk loading
- High-throughput ingestion pipeline
- Error handling with retries
- Rate limiting
- Throughput monitoring

In [ ]:
from hindsight_client import Hindsight
import asyncio
import time
from datetime import datetime, timezone

HINDSIGHT_URL = "http://localhost:8888"
BANK_ID = "tutorial-async"

# Create a fresh bank
sync_client = Hindsight(base_url=HINDSIGHT_URL)
sync_client.create_bank(
    bank_id=BANK_ID,
    name="Async Tutorial",
    reflect_mission="I handle high-throughput memory operations."
)
print(f"✓ Bank '{BANK_ID}' ready")

## 1. Async Client Basics

All operations have async equivalents prefixed with `a`.

In [ ]:
async def async_basics():
    """Demonstrate basic async operations."""
    client = Hindsight(base_url=HINDSIGHT_URL)
    
    # Async retain
    await client.aretain(
        bank_id=BANK_ID,
        content="Async test: Alice works at Google"
    )
    print("✓ aretain() done")
    
    # Async recall
    results = await client.arecall(
        bank_id=BANK_ID,
        query="Alice"
    )
    print(f"✓ arecall() returned {len(results.results)} results")
    
    # Async reflect
    answer = await client.areflect(
        bank_id=BANK_ID,
        query="Where does Alice work?"
    )
    print(f"✓ areflect() answer: {answer.text[:100]}...")
    
    await client.aclose()
    print("✓ Client closed")

await async_basics()

## 2. Async Context Manager

In [ ]:
async def async_context_manager():
    """Use async context manager for automatic cleanup."""
    async with Hindsight(base_url=HINDSIGHT_URL) as client:
        await client.aretain(
            bank_id=BANK_ID,
            content="Context manager test: Bob works at Meta"
        )
        
        results = await client.arecall(
            bank_id=BANK_ID,
            query="Bob"
        )
        print(f"✓ Found {len(results.results)} results for Bob")
    # Auto-closed after exiting the 'async with' block
    print("✓ Auto-closed via context manager")

await async_context_manager()

## 3. Concurrent Operations

Run multiple retains or recalls in parallel using `asyncio.gather`.

In [ ]:
async def concurrent_retains():
    """Fire multiple retains in parallel."""
    async with Hindsight(base_url=HINDSIGHT_URL) as client:
        items = [
            "Carol is a data engineer at Amazon",
            "Dave is a product manager at Stripe",
            "Eve is a designer at Figma",
            "Frank is a researcher at DeepMind",
            "Grace is an SRE at Netflix",
        ]
        
        start = time.monotonic()
        
        # Run all retains concurrently
        tasks = [
            client.aretain(bank_id=BANK_ID, content=item)
            for item in items
        ]
        results = await asyncio.gather(*tasks)
        
        elapsed = (time.monotonic() - start) * 1000
        print(f"✓ {len(items)} concurrent retains in {elapsed:.0f}ms")
        print(f"  vs sequential: ~{len(items) * 200}ms estimated")

await concurrent_retains()

In [ ]:
async def concurrent_recalls():
    """Search across multiple banks in parallel."""
    async with Hindsight(base_url=HINDSIGHT_URL) as client:
        # Create a few banks for demonstration
        for i in range(3):
            bid = f"{BANK_ID}-{i}"
            try:
                client.create_bank(bank_id=bid, name=f"Sub-bank {i}")
                client.retain(bank_id=bid, content=f"Sub-bank {i} has Python preference")
            except Exception:
                pass
        
        # Search all banks in parallel
        bank_ids = [f"{BANK_ID}-{i}" for i in range(3)]
        
        tasks = [
            client.arecall(bank_id=bid, query="Python", budget="low")
            for bid in bank_ids
        ]
        
        all_results = await asyncio.gather(*tasks)
        
        for bid, results in zip(bank_ids, all_results):
            print(f"  {bid}: {len(results.results)} results")
        
        # Cleanup sub-banks
        for bid in bank_ids:
            try:
                client.delete_bank(bank_id=bid)
            except Exception:
                pass

await concurrent_recalls()

## 4. Batch Retain — Bulk Loading

`retain_batch()` processes multiple items in a single LLM call.

In [ ]:
async def batch_loading():
    """Demonstrate batch retain vs individual retains."""
    async with Hindsight(base_url=HINDSIGHT_URL) as client:
        # Prepare batch items
        batch_items = [
            {"content": f"Batch test #{i}: The API processed this in {i*10}ms"}
            for i in range(20)
        ]
        
        start = time.monotonic()
        
        # Batch retain — one LLM call for all items
        result = await client.aretain_batch(
            bank_id=BANK_ID,
            items=batch_items,
            document_id="batch-test-001",
            retain_async=False
        )
        
        elapsed = (time.monotonic() - start) * 1000
        print(f"✓ Batch: {result.items_count} items in {elapsed:.0f}ms")
        print(f"  vs ~{20 * 200}ms if done individually")

In [ ]:
await batch_loading()

## 5. Async Retain — Fire and Forget

With `retain_async=True`, the call returns immediately.

In [ ]:
async def fire_and_forget():
    """Demonstrate fire-and-forget async retains."""
    async with Hindsight(base_url=HINDSIGHT_URL) as client:
        start = time.monotonic()
        
        # Fire 50 retains — all return immediately
        tasks = [
            client.aretain(
                bank_id=BANK_ID,
                content=f"Log entry #{i}: request completed",
                context="metrics",
                retain_async=True  # ← Fire and forget
            )
            for i in range(50)
        ]
        
        results = await asyncio.gather(*tasks)
        
        elapsed = (time.monotonic() - start) * 1000
        print(f"✓ Submitted {len(results)} async retains in {elapsed:.0f}ms")
        print(f"  Each returns: {{'success': True, 'operation_id': 'op_...'}}")
        
        # Check queued operations
        await asyncio.sleep(1)
        stats = client.get_bank_stats(bank_id=BANK_ID)
        pending = stats.get('pending_operations', '?')
        print(f"  Pending operations: {pending}")

In [ ]:
await fire_and_forget()

## 6. High-Throughput Ingestion Pipeline

A production-ready buffered ingestion pipeline.

In [ ]:
from asyncio import Queue

class MemoryIngestor:
    """Buffered, batched memory ingestion pipeline.
    
    Collects items in a queue and flushes them in batches
    at regular intervals for maximum throughput.
    """
    
    def __init__(self, base_url, bank_id, batch_size=50, flush_interval=1.0):
        self.base_url = base_url
        self.bank_id = bank_id
        self.batch_size = batch_size
        self.flush_interval = flush_interval
        self.queue = Queue()
        self.total_ingested = 0
        self.total_flushed = 0
        self.client = None
    
    async def start(self):
        """Initialize the client and start the flush loop."""
        self.client = Hindsight(base_url=self.base_url)
        self._running = True
    
    async def ingest(self, content, **kwargs):
        """Add an item to the ingestion queue."""
        item = {"content": content, **kwargs}
        await self.queue.put(item)
        self.total_ingested += 1
    
    async def flush(self):
        """Flush the queue as a batch."""
        batch = []
        while not self.queue.empty() and len(batch) < self.batch_size:
            try:
                item = self.queue.get_nowait()
                batch.append(item)
            except asyncio.QueueEmpty:
                break
        
        if batch:
            try:
                result = await self.client.aretain_batch(
                    bank_id=self.bank_id,
                    items=batch,
                    retain_async=False
                )
                self.total_flushed += len(batch)
                return len(batch)
            except Exception as e:
                print(f"  Batch flush failed: {e}")
                # Re-queue items on failure
                for item in batch:
                    await self.queue.put(item)
                return 0
        return 0
    
    async def run(self, duration_seconds=5):
        """Run the flush loop for a specified duration."""
        deadline = time.monotonic() + duration_seconds
        
        while time.monotonic() < deadline:
            flushed = await self.flush()
            await asyncio.sleep(self.flush_interval)
        
        # Final flush
        await self.flush()
    
    async def shutdown(self):
        """Final flush and cleanup."""
        await self.flush()
        if self.client:
            await self.client.aclose()
        print(f"  Total ingested: {self.total_ingested}")
        print(f"  Total flushed: {self.total_flushed}")

# Usage
async def run_pipeline():
    ingestor = MemoryIngestor(
        HINDSIGHT_URL, BANK_ID,
        batch_size=30,
        flush_interval=0.5
    )
    await ingestor.start()
    
    # Simulate high-volume ingestion
    start = time.monotonic()
    for i in range(200):
        await ingestor.ingest(
            content=f"Pipeline event #{i}: processed",
            context="pipeline",
            metadata={"batch": str(i // 30)}
        )
    
    ingest_time = (time.monotonic() - start) * 1000
    print(f"✓ Queued {ingestor.total_ingested} items in {ingest_time:.0f}ms")
    
    # Run flush loop
    await ingestor.run(duration_seconds=3)
    await ingestor.shutdown()

await run_pipeline()

## 7. Error Handling with Retries

In [ ]:
# Simple retry decorator using tenacity (or manual)
async def robust_retain(client, bank_id, content, max_retries=3):
    """Retain with exponential backoff on failure."""
    for attempt in range(max_retries):
        try:
            result = await client.aretain(
                bank_id=bank_id,
                content=content
            )
            return result
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"  ✗ Failed after {max_retries} attempts: {e}")
                raise
            wait = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
            print(f"  ⚠ Attempt {attempt + 1} failed, retrying in {wait}s...")
            await asyncio.sleep(wait)

async def test_retry():
    async with Hindsight(base_url=HINDSIGHT_URL) as client:
        try:
            result = await robust_retain(
                client, BANK_ID,
                "Critical fact that must be stored"
            )
            print("✓ Retain succeeded with retry logic")
        except Exception:
            print("✗ All retries exhausted")

await test_retry()

## 8. Rate Limiter

In [ ]:
class TokenBucketRateLimiter:
    """Token bucket rate limiter for API calls."""
    
    def __init__(self, max_per_second):
        self.rate = max_per_second
        self.tokens = max_per_second
        self.last_refill = time.monotonic()
    
    async def acquire(self):
        """Wait until a token is available."""
        while True:
            now = time.monotonic()
            elapsed = now - self.last_refill
            self.tokens = min(self.rate, self.tokens + elapsed * self.rate)
            self.last_refill = now
            
            if self.tokens >= 1:
                self.tokens -= 1
                return
            
            await asyncio.sleep(1.0 / self.rate)

async def rate_limited_work():
    """Demonstrate rate-limited retains."""
    limiter = TokenBucketRateLimiter(max_per_second=5)
    
    async with Hindsight(base_url=HINDSIGHT_URL) as client:
        start = time.monotonic()
        
        for i in range(10):
            await limiter.acquire()
            await client.aretain(
                bank_id=BANK_ID,
                content=f"Rate-limited event #{i}",
                retain_async=True
            )
        
        elapsed = time.monotonic() - start
        print(f"✓ 10 retains at 5/sec took {elapsed:.1f}s (expected ~2s)")

In [ ]:
await rate_limited_work()

## 9. Throughput Monitor

In [ ]:
from collections import defaultdict

class ThroughputMonitor:
    """Track latency and error rates for Hindsight operations."""
    
    def __init__(self):
        self.stats = defaultdict(lambda: {"count": 0, "total_ms": 0.0, "errors": 0})
    
    async def timed_retain(self, client, bank_id, content):
        start = time.monotonic()
        try:
            result = await client.aretain(bank_id=bank_id, content=content)
            elapsed = (time.monotonic() - start) * 1000
            self.stats["retain"]["count"] += 1
            self.stats["retain"]["total_ms"] += elapsed
            return result
        except Exception:
            self.stats["retain"]["errors"] += 1
            raise
    
    async def timed_recall(self, client, bank_id, query):
        start = time.monotonic()
        try:
            result = await client.arecall(bank_id=bank_id, query=query)
            elapsed = (time.monotonic() - start) * 1000
            self.stats["recall"]["count"] += 1
            self.stats["recall"]["total_ms"] += elapsed
            return result
        except Exception:
            self.stats["recall"]["errors"] += 1
            raise
    
    def report(self):
        print("=== Throughput Report ===")
        for op, data in sorted(self.stats.items()):
            count = data["count"]
            avg = data["total_ms"] / count if count > 0 else 0
            errors = data["errors"]
            print(f"  {op:10s}: {count:4d} calls | avg {avg:6.1f}ms | {errors} errors")

async def monitor_demo():
    monitor = ThroughputMonitor()
    
    async with Hindsight(base_url=HINDSIGHT_URL) as client:
        for i in range(5):
            await monitor.timed_retain(client, BANK_ID, f"Monitored event #{i}")
        
        for query in ["test", "Alice", "Bob"]:
            await monitor.timed_recall(client, BANK_ID, query)
    
    monitor.report()

await monitor_demo()

## 10. Cleanup

In [ ]:
# sync_client.delete_bank(bank_id=BANK_ID)
print("Bank preserved.")

## Summary

You learned:
1. **Async client** — all operations have `a`-prefixed versions
2. **Context manager** — `async with` for automatic cleanup
3. **Concurrent operations** — `asyncio.gather` for parallel retains/recalls
4. **Batch retain** — single LLM call for multiple items (much faster)
5. **Fire and forget** — `retain_async=True` for immediate return
6. **Ingestion pipeline** — buffered, batched processing for high throughput
7. **Retry logic** — exponential backoff on failures
8. **Rate limiting** — token bucket algorithm
9. **Throughput monitoring** — latency and error tracking

**Next:** [09_mcp_integration.ipynb](09_mcp_integration.ipynb) — MCP integration